# Cell type composition analysis

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 

            # Shannon diversity 
            library(vegan), 

            # Data 
            library(tidyverse), 
            library(data.table), 

            # Plotting 
            library(patchwork), 
            library(cowplot), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [ ]:
options(warn=-1)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
source("bin/seurat_qc.R")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Import Seurat 

In [ ]:
so <- readRDS("data/object/pp.rds")

In [ ]:
options(repr.plot.width=10, repr.plot.height=10)

dplot_1 <- dplot(so, reduction="umap", group_by="celltype_low", alpha=1, pt_size=1.5, label=TRUE, label_size=16, label_box=TRUE, label_box_parse=TRUE, label_text_color="white") + 
    ggtitle("") + 
    scale_color_manual(values=color$celltype_low_r, na.value="grey50") + 
    scale_fill_manual(values=color$celltype_low_r, na.value="grey50") + 
    theme(legend.position="none")

dplot_1

# Cell type distribution 

In [ ]:
table(so$gvhd_status)

In [ ]:
# Meta data
meta <- dplyr::select(so@meta.data, sample_name, celltype_low, patient_id, time_point, hto_demux_class, gvhd_status) %>% 
    dplyr::mutate(sample_group=paste0(sample_name, "_", hto_demux_class)) %>% 
    dplyr::mutate(patient_id=gsub("_", " ", patient_id)) %>% 
    unique()

In [ ]:
# Cell count data 
mat_count <- dplyr::select(so@meta.data, sample_name, hto_demux_class, celltype_low) %>% 
    dplyr::mutate(sample_group=paste0(sample_name, "_", hto_demux_class)) %>% 
    dplyr::group_by(sample_group, celltype_low) %>% 
    dplyr::summarise(count=n()) %>% 
    dplyr::ungroup()

In [ ]:
# Combine counts with meta data 
meta <- dplyr::left_join(mat_count, meta, by=join_by(sample_group, celltype_low))

In [ ]:
# Transform mat count to cell type x sample group format
mat_count <- reshape2::dcast(mat_count, celltype_low~sample_group, value.var="count") %>% tibble::column_to_rownames("celltype_low") %>% replace(is.na(.), 0)

In [ ]:
# Library normalizing factors 
mat_norm <- edgeR::DGEList(counts=mat_count)
mat_norm <- edgeR::calcNormFactors(mat_norm)

In [ ]:
# Compute log2(CPM)
mat_norm <- edgeR::cpm(mat_norm, log=TRUE, normalized.lib.sizes=TRUE, prior.count=0)

In [ ]:
# Combine log2(CPM) with cell count meta
mat_norm <- reshape2::melt(mat_norm, varnames=c("celltype_low", "sample_group"), value.name="log2_cpm")
meta <- dplyr::left_join(mat_norm, meta, by=join_by(sample_group, celltype_low))

In [ ]:
# Filter meta for samples with counts 
meta <- meta[!is.na(meta$sample_name), ]

In [ ]:
plot_1 <- ggplot(meta[meta$hto_demux_class=="Blood", ], aes(x=patient_id, y=log2_cpm, fill=celltype_low)) + 
    geom_bar(stat="identity") + 
    ggtitle("") + xlab("Patient ID") + ylab("Cell type [log2(CPM)]")+ 
    scale_fill_manual(values=color$celltype_low_r, name="Cell type", labels=scales::parse_format()) +
    facet_grid(~time_point, scales="free", space="free") +
    theme(axis.text.x = element_text(angle=90, vjust=0.5, hjust=1)) +  
    guides(fill=guide_legend(ncol=1, nrow=length(color$celltype_low_r)))

plot_2 <- ggplot(meta[meta$hto_demux_class=="Blood", ], aes(x=patient_id, y=log2_cpm, fill=celltype_low)) + 
    geom_bar(stat="identity", position="fill") + 
    ggtitle("") + xlab("Patient ID") + ylab("Cell type [Ratio log2(CPM)]")+ 
    scale_fill_manual(values=color$celltype_low_r, name="Cell type", labels=scales::parse_format()) +
    facet_grid(~time_point, scales="free", space="free") + 
    theme(axis.text.x = element_text(angle=90, vjust=0.5, hjust=1)) + 
    guides(fill=guide_legend(ncol=1, nrow=length(color$celltype_low_r)))

In [ ]:
plot_3 <- ggplot(meta[meta$hto_demux_class=="Skin", ], aes(x=patient_id, y=log2_cpm, fill=celltype_low)) + 
    geom_bar(stat="identity") + 
    ggtitle("") + xlab("Patient ID") + ylab("Cell type [log2(CPM)]")+ 
    scale_fill_manual(values=color$celltype_low_r, name="Cell type", labels=scales::parse_format()) +
    facet_grid(~time_point, scales="free", space="free") +
    theme(axis.text.x = element_text(angle=90, vjust=0.5, hjust=1)) +  
    guides(fill=guide_legend(ncol=2, nrow=length(color$celltype_low_r)))

plot_4 <- ggplot(meta[meta$hto_demux_class=="Skin", ], aes(x=patient_id, y=log2_cpm, fill=celltype_low)) + 
    geom_bar(stat="identity", position="fill") + 
    ggtitle("") + xlab("Patient ID") + ylab("Cell type [Ratio log2(CPM)]")+ 
    scale_fill_manual(values=color$celltype_low_r, name="Cell type", labels=scales::parse_format()) +
    facet_grid(~time_point, scales="free", space="free") + 
    theme(axis.text.x = element_text(angle=90, vjust=0.5, hjust=1)) + 
    guides(fill=guide_legend(ncol=2, nrow=length(color$celltype_low_r)))

In [ ]:
options(repr.plot.width=20, repr.plot.height=10)

plot_list <- list(
    
    plot_1 + theme(legend.position="none"), 
    plot_2 + theme(legend.position="none"), 
    plot_3 + theme(legend.position="none"), 
    plot_4 + theme(legend.position="none")
)
legend <- cowplot::get_legend(plot_1 + plot_4 + patchwork::plot_layout(ncol=1, guides="collect"))

cowplot::plot_grid(plotlist=plot_list, ncol=2, align="v") %>% cowplot::plot_grid(., legend, ncol=2, rel_widths = c(3, 0.75))

# Shannon entropy

## Time points by sample 

In [ ]:
cell_count <- dplyr::mutate(so@meta.data, sample_group=paste(sample_name, celltype_domain, hto_demux_class)) %>%
    dplyr::group_by(sample_group, sample_name, patient_id, celltype_domain, hto_demux_class, time_point, celltype_low) %>% 
    dplyr::summarise(cell_count=n()) %>% dplyr::ungroup()

cell_count_cnt <- dcast(as.data.table(cell_count), celltype_low~sample_group, value.var="cell_count") %>% tibble::column_to_rownames("celltype_low") %>% replace(is.na(.), 0) 

In [ ]:
shannon <- data.frame(sample_group=colnames(cell_count_cnt), shannon=vegan::diversity(t(cell_count_cnt), index="invsimpson", base=2)) %>% dplyr::left_join(., dplyr::select(cell_count, sample_group, celltype_domain, hto_demux_class, patient_id, time_point, sample_name) %>% unique(), by="sample_group")

In [ ]:
options(repr.plot.width=10, repr.plot.height=3)

plot_1 <- ggplot(shannon[shannon$hto_demux_class=="Blood", ], aes(x=time_point, y=shannon)) + 
    geom_boxplot(aes(fill=time_point), outlier.shape=1, outlier.size=3, size=0.5, color="black") + 
    geom_point(size=3) + 
    ggtitle("Blood") + xlab("") + ylab("Shannon") +
    ylim(0, max(shannon$shannon)+0.1) +
    scale_fill_manual(values=color$time_point, name="Time point")

plot_2 <- ggplot(shannon[shannon$hto_demux_class=="Skin" & shannon$celltype_domain=="Immune~cell", ], aes(x=time_point, y=shannon)) + 
    geom_boxplot(aes(fill=time_point), outlier.shape=1, outlier.size=3, size=0.5, color="black") + 
    geom_point(size=3) + 
    ggtitle("Skin (Immune)") + xlab("") + ylab("Shannon") +
    ylim(0, max(shannon$shannon)+0.1) +
    scale_fill_manual(values=color$time_point, name="Time point") +
    scale_shape_manual(values=setNames(c(1, 2), c("GVHD free", "GVHD dev.")), name="GVHD class")

plot_3 <- ggplot(shannon[shannon$hto_demux_class=="Skin" & shannon$celltype_domain=="'Non-hematopoetic'", ], aes(x=time_point, y=shannon)) + 
    geom_boxplot(aes(fill=time_point), outlier.shape=1, outlier.size=3, size=0.5, color="black") + 
    geom_point(size=3) + 
    ggtitle("Skin (Non immune)") + xlab("") + ylab("Shannon") +
    ylim(0, max(shannon$shannon)+0.1) +
    scale_fill_manual(values=color$time_point, name="Time point") +
    scale_shape_manual(values=setNames(c(1, 2), c("GVHD free", "GVHD dev.")), name="GVHD class")

## Time points by sample and genotype 

In [ ]:
cell_count <- dplyr::mutate(so@meta.data, sample_group=paste(sample_name, genotype_class, celltype_domain, hto_demux_class)) %>%
    dplyr::group_by(sample_group, sample_name, genotype_class, celltype_domain, hto_demux_class, time_point, celltype_low) %>%  
    dplyr::summarise(cell_count=n())

In [ ]:
cell_count_cnt <- dcast(as.data.table(cell_count), celltype_low~sample_group, value.var="cell_count") %>% tibble::column_to_rownames("celltype_low") %>% replace(is.na(.), 0)

In [ ]:
shannon <- data.frame(sample_group=colnames(cell_count_cnt), shannon=vegan::diversity(t(cell_count_cnt), index="invsimpson", base=2)) %>% 
    dplyr::left_join(., dplyr::select(cell_count, sample_group, sample_name, genotype_class, celltype_domain, hto_demux_class, time_point) %>% unique(), by="sample_group") %>% 
    dplyr::group_by(genotype_class, celltype_domain, hto_demux_class, time_point) %>% 
    dplyr::summarise(shannon_median=median(shannon), shannon_mad=mad(shannon, constant=1)) %>%
    dplyr::ungroup() %>% 
    tidyr::complete(genotype_class, celltype_domain, hto_demux_class, time_point, fill=list(shannon_median=0, shannon_mad=0)) %>% 
    dplyr::filter(genotype_class %in% c("Host", "Donor"))

In [ ]:
options(repr.plot.width=10, repr.plot.height=3)

dodge_with <- 0.5
mad_scale <- 2

plot_4 <- ggplot(shannon[shannon$hto_demux_class=="Blood" & shannon$celltype_domain=="Immune~cell", ], aes(x=time_point, y=shannon_median, color=genotype_class)) + 
    geom_line(aes(group=genotype_class), size=1, position=position_dodge(dodge_with)) + 
    geom_point(shape=16, size=3, position=position_dodge(dodge_with)) + 
    geom_errorbar(aes(ymin=ifelse(shannon_median-mad_scale*shannon_mad<0, 0, shannon_median-mad_scale*shannon_mad), ymax=shannon_median+mad_scale*shannon_mad), width=0.5, size=0.75, position=position_dodge(dodge_with)) + 
    ggtitle("Blood") + xlab("") + ylab("Shannon") +
    # ylim(0, 1) +
    scale_color_manual(values=color$genotype_class, name="Genotype class") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

plot_5 <- ggplot(shannon[shannon$hto_demux_class=="Skin" & shannon$celltype_domain=="Immune~cell", ], aes(x=time_point, y=shannon_median, color=genotype_class)) + 
    geom_line(aes(group=genotype_class), size=1, position=position_dodge(dodge_with)) + 
    geom_point(shape=16, size=3, position=position_dodge(dodge_with)) + 
    geom_errorbar(aes(ymin=ifelse(shannon_median-mad_scale*shannon_mad<0, 0, shannon_median-mad_scale*shannon_mad), ymax=shannon_median+mad_scale*shannon_mad), width=0.5, size=0.75, position=position_dodge(dodge_with)) + 
    ggtitle("Skin (Immune)") + xlab("") + ylab("Shannon") +
    # ylim(0, 1) +
    scale_color_manual(values=color$genotype_class, name="Genotype class") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

plot_6 <- ggplot(shannon[shannon$hto_demux_class=="Skin" & shannon$celltype_domain=="'Non-hematopoetic'", ], aes(x=time_point, y=shannon_median, color=genotype_class)) + 
    geom_line(aes(group=genotype_class), size=1, position=position_dodge(dodge_with)) + 
    geom_point(shape=16, size=3, position=position_dodge(dodge_with)) + 
    geom_errorbar(aes(ymin=ifelse(shannon_median-mad_scale*shannon_mad<0, 0, shannon_median-mad_scale*shannon_mad), ymax=shannon_median+mad_scale*shannon_mad), width=0.5, size=0.75, position=position_dodge(dodge_with)) + 
    ggtitle("Skin (Non immune)") + xlab("") + ylab("Shannon") +
    # ylim(0, 1) +
    scale_color_manual(values=color$genotype_class, name="Genotype class") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

## Genotype ratio by time points 

In [ ]:
genotype_ratio <- dplyr::group_by(so@meta.data, celltype_domain, patient_id, hto_demux_class, time_point, genotype_class) %>%  
    dplyr::summarise(cell_count=n()) %>% dplyr::ungroup() %>% 
    tidyr::complete(celltype_domain, patient_id, hto_demux_class, time_point, genotype_class, fill=list(cell_count=0)) %>% 
    dplyr::group_by(celltype_domain, patient_id, hto_demux_class, time_point) %>% 
    dplyr::mutate(library_size=sum(cell_count)) %>% dplyr::ungroup() %>% 
    dplyr::filter(genotype_class %in% c("Host", "Donor")) %>% 
    dplyr::mutate(ratio=cell_count/library_size) %>% 
    dplyr::group_by(celltype_domain, hto_demux_class, time_point, genotype_class) %>% 
    dplyr::summarise(ratio_mean=mean(ratio, na.rm=TRUE), ratio_sd=sd(ratio, na.rm=TRUE))

In [ ]:
options(repr.plot.width=10, repr.plot.height=3)

dodge_with <- 0.5

plot_7 <- ggplot(genotype_ratio[genotype_ratio$hto_demux_class=="Blood" & genotype_ratio$celltype_domain=="Immune~cell", ], aes(x=time_point, y=ratio_mean, color=genotype_class)) + 
    geom_line(aes(group=genotype_class), size=1, position=position_dodge(dodge_with)) + 
    geom_point(shape=16, size=3, position=position_dodge(dodge_with)) + 
    geom_errorbar(aes(ymin=ifelse(ratio_mean-ratio_sd<0, 0, ratio_mean-ratio_sd), ymax=ifelse(ratio_mean+ratio_sd>1, 1, ratio_mean+ratio_sd)), width=0.5, size=0.75, position=position_dodge(dodge_with)) + 
    ggtitle("Blood") + xlab("") + ylab("Ratio") +
    ylim(0, 1) +
    scale_color_manual(values=color$genotype_class, name="Genotype class") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

plot_8 <- ggplot(genotype_ratio[genotype_ratio$hto_demux_class=="Skin" & genotype_ratio$celltype_domain=="Immune~cell", ], aes(x=time_point, y=ratio_mean, color=genotype_class)) + 
    geom_line(aes(group=genotype_class), size=1, position=position_dodge(dodge_with)) + 
    geom_point(shape=16, size=3, position=position_dodge(dodge_with)) + 
    geom_errorbar(aes(ymin=ifelse(ratio_mean-ratio_sd<0, 0, ratio_mean-ratio_sd), ymax=ifelse(ratio_mean+ratio_sd>1, 1, ratio_mean+ratio_sd)), width=0.5, size=0.75, position=position_dodge(dodge_with)) + 
    ggtitle("Skin (Immune)") + xlab("") + ylab("Ratio") +
    ylim(0, 1) +
    scale_color_manual(values=color$genotype_class, name="Genotype class") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

plot_9 <- ggplot(genotype_ratio[genotype_ratio$hto_demux_class=="Skin" & genotype_ratio$celltype_domain=="'Non-hematopoetic'", ], aes(x=time_point, y=ratio_mean, color=genotype_class)) + 
    geom_line(aes(group=genotype_class), size=1, position=position_dodge(dodge_with)) + 
    geom_point(shape=16, size=3, position=position_dodge(dodge_with)) + 
    geom_errorbar(aes(ymin=ifelse(ratio_mean-ratio_sd<0, 0, ratio_mean-ratio_sd), ymax=ifelse(ratio_mean+ratio_sd>1, 1, ratio_mean+ratio_sd)), width=0.5, size=0.75, position=position_dodge(dodge_with)) + 
    ggtitle("Skin (Non immune)") + xlab("") + ylab("Ratio") +
    ylim(0, 1) +
    scale_color_manual(values=color$genotype_class, name="Genotype class") + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

In [ ]:
options(repr.plot.width=10, repr.plot.height=7.5)

plot_list <- list(
    
    plot_1 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank()), 
    plot_2 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank()), 
    plot_3 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank()), 
    plot_4 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank(), plot.title=element_blank()), 
    plot_5 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank(), plot.title=element_blank()), 
    plot_6 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank(), plot.title=element_blank()),  
    plot_7 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank(), plot.title=element_blank()),  
    plot_8 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank(), plot.title=element_blank()),  
    plot_9 + theme(legend.position="none", axis.text.x=element_blank(), axis.title.x=element_blank(), plot.title=element_blank())

)

legend <- cowplot::get_legend(plot_1 + plot_4 + patchwork::plot_layout(ncol=1, guides="collect"))

plot <- cowplot::plot_grid(plotlist=plot_list, ncol=3, nrow=3, align="vh", rel_heights=c(1.5, 1, 1)) %>% cowplot::plot_grid(., legend, ncol=2, rel_widths = c(3, 0.75)) 
plot

In [ ]:
# ggsave("result/composition/figure/composition_time_point_genotype.png", plot, dpi=1300, width=13.5, height=7.5, units="in")

# Info

In [ ]:
sessionInfo()